# MADDPG — 1.5M (Modal A100)
**Environment:** `simple_spread_v3` — N=3 agents, continuous actions, max_cycles=50

**Algorithm:** MADDPG — Multi-Agent Deep Deterministic Policy Gradient (CTDE)
- Per-agent actor: local obs → continuous action (5-dim) in [0,1] via Sigmoid
- Per-agent centralised critic: global obs + all agents actions → Q-value
- Off-policy: experience replay buffer (capacity 1,000,000)
- Target networks with soft update (τ=0.01)
- Ornstein-Uhlenbeck noise for exploration (clipped to [0,1])
- Training runs on **Modal A100 GPU** — safe to close browser after submission
- Results auto-saved to Google Drive

## Fixes applied:
- Sigmoid (not Tanh) on actor output — matches Box(0.0, 1.0) action space
- OU noise centred at mu=0.5 (midpoint of [0,1])
- Action clip to [0.0, 1.0] (not [-1,1])
- Vectorised batch normalisation — no Python loops inside update


In [ ]:
# CELL 1 — Install
!pip install -q \
    modal \
    pettingzoo==1.24.3 \
    gymnasium==0.29.1 \
    supersuit==3.9.3 \
    torch numpy matplotlib

In [ ]:
# CELL 2 — Authenticate Modal

import os, subprocess, sys

os.environ["MODAL_TOKEN_ID"]     = "ak-qzTQRLuRQxGA8tZkVEIJwu"   # ← your Token ID
os.environ["MODAL_TOKEN_SECRET"] = "as-NXte5gutCbH8XMMj2K75zI"   # ← your Token Secret

config_dir = os.path.expanduser("~/.modal")
os.makedirs(config_dir, exist_ok=True)
with open(f"{config_dir}/config.toml", "w") as f:
    f.write(f'[default]\ntoken_id = "{os.environ["MODAL_TOKEN_ID"]}"\n'
            f'token_secret = "{os.environ["MODAL_TOKEN_SECRET"]}"\n')

r = subprocess.run(
    [sys.executable, "-m", "modal", "token", "set",
     "--token-id",     os.environ["MODAL_TOKEN_ID"],
     "--token-secret", os.environ["MODAL_TOKEN_SECRET"]],
    capture_output=True, text=True
)
print("✅ Authenticated" if r.returncode == 0 else f"❌ Failed: {r.stderr}")
print("Modal authentication complete — ready to submit training job")

print("\n✅ Authentication done")
print("📌 IMPORTANT: Cell 18 uses --detach flag")
print("   This means Modal keeps running even if Colab disconnects")
print("   You can safely close this tab after submitting")

✅ Authenticated
Modal authentication complete — ready to submit training job

✅ Authentication done
📌 IMPORTANT: Cell 18 uses --detach flag
   This means Modal keeps running even if Colab disconnects
   You can safely close this tab after submitting


In [ ]:
# CELL 3 — GPU Check (Modal A100)
import torch
print(f"Torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
print("Note: Training will run on Modal A100 — not this Colab session")
print("This Colab is just used to submit the job")

Torch  : 2.11.0+cpu
CUDA   : False
Note: Training will run on Modal A100 — not this Colab session
This Colab is just used to submit the job


In [ ]:
# CELL 4 — Write Modal training script to disk
import base64, os

script_b64 = "..."
script = base64.b64decode(script_b64).decode()
with open('/content/maddpg_modal.py', 'w') as f:
    f.write(script)

size = os.path.getsize('/content/maddpg_modal.py')
print(f"Modal training script written — {size:,} bytes")
print("Uses .spawn() — safe to disconnect after submitting")

Modal training script written — 17,249 bytes
Uses .spawn() — safe to disconnect after submitting


In [ ]:
# CELL 5 — Environment config (for reference)
# These are set inside the Modal script — shown here for reference
print("Environment : simple_spread_v3")
print("Agents      : 3")
print("Obs dim     : 18")
print("Action dim  : 5 (continuous, Sigmoid output)")
print("Action space: Box(0.0, 1.0, (5,), float32)")
print("Global state: 54-dim (all agents concat)")
print("Global acts : 15-dim (all agents concat)")

Environment : simple_spread_v3
Agents      : 3
Obs dim     : 18
Action dim  : 5 (continuous, Sigmoid output)
Action space: Box(0.0, 1.0, (5,), float32)
Global state: 54-dim (all agents concat)
Global acts : 15-dim (all agents concat)


In [ ]:
# CELL 6 — Actor architecture (for reference)
# Sigmoid output ensures actions stay in [0,1]
print("Actor architecture:")
print("  Linear(18→256) → LayerNorm → ReLU")
print("  Linear(256→256) → LayerNorm → ReLU")
print("  Linear(256→5) → Sigmoid  ← [0,1] output")
print()
print("Critic architecture:")
print("  Input: 54-dim state + 15-dim all actions = 69-dim")
print("  Linear(69→256) → LayerNorm → ReLU")
print("  Linear(256→256) → LayerNorm → ReLU")
print("  Linear(256→1)")

Actor architecture:
  Linear(18→256) → LayerNorm → ReLU
  Linear(256→256) → LayerNorm → ReLU
  Linear(256→5) → Sigmoid  ← [0,1] output

Critic architecture:
  Input: 54-dim state + 15-dim all actions = 69-dim
  Linear(69→256) → LayerNorm → ReLU
  Linear(256→256) → LayerNorm → ReLU
  Linear(256→1)


In [ ]:
# CELL 7 — Networks (for reference)
print("Per-agent networks:")
print("  3 x Actor         (separate per agent)")
print("  3 x Critic        (separate per agent)")
print("  3 x Target Actor  (soft update tau=0.01)")
print("  3 x Target Critic (soft update tau=0.01)")
print("  Total: 12 networks")

Per-agent networks:
  3 x Actor         (separate per agent)
  3 x Critic        (separate per agent)
  3 x Target Actor  (soft update tau=0.01)
  3 x Target Critic (soft update tau=0.01)
  Total: 12 networks


In [ ]:
# CELL 8 — Hyperparameters (1.5M)
print("Hyperparameters — 1.5M run:")
print(f"  TOTAL_STEPS      : 1,500,000")
print(f"  GAMMA            : 0.99")
print(f"  TAU              : 0.01")
print(f"  BATCH_SIZE       : 512")
print(f"  BUFFER_CAPACITY  : 1,000,000")
print(f"  UPDATE_EVERY     : 2")
print(f"  ACTOR_LR         : 5e-5")
print(f"  CRITIC_LR        : 1e-4")
print(f"  OU_SIGMA start   : 0.15  → min 0.05")
print(f"  WARMUP_STEPS     : 5,000")
print(f"  GPU              : Modal A100")
print(f"  Expected time    : ~4-5 hours")
print(f"  Expected cost    : ~$10 from $30 free credits")

Hyperparameters — 1.5M run:
  TOTAL_STEPS      : 1,500,000
  GAMMA            : 0.99
  TAU              : 0.01
  BATCH_SIZE       : 512
  BUFFER_CAPACITY  : 1,000,000
  UPDATE_EVERY     : 2
  ACTOR_LR         : 5e-5
  CRITIC_LR        : 1e-4
  OU_SIGMA start   : 0.15  → min 0.05
  WARMUP_STEPS     : 5,000
  GPU              : Modal A100
  Expected time    : ~4-5 hours
  Expected cost    : ~$10 from $30 free credits


In [ ]:
# CELL 9 — Optimizers (for reference)
print("Optimizers:")
print("  Actor  : Adam  lr=5e-5  eps=1e-5  (per agent)")
print("  Critic : Adam  lr=1e-4  eps=1e-5  (per agent)")

Optimizers:
  Actor  : Adam  lr=5e-5  eps=1e-5  (per agent)
  Critic : Adam  lr=1e-4  eps=1e-5  (per agent)


In [ ]:
# CELL 10 — Replay Buffer (for reference)
print("Replay Buffer:")
print("  Capacity : 1,000,000 transitions")
print("  Stores   : (obs, next_obs, actions, per_obs, per_nobs, reward, done)")
print("  Sampling : random mini-batch of 512")
print("  Type     : off-policy (MADDPG)")

Replay Buffer:
  Capacity : 1,000,000 transitions
  Stores   : (obs, next_obs, actions, per_obs, per_nobs, reward, done)
  Sampling : random mini-batch of 512
  Type     : off-policy (MADDPG)


In [ ]:
# CELL 11 — OU Noise (for reference)
print("Ornstein-Uhlenbeck Noise:")
print("  mu    : 0.5  (centred for [0,1] action space)")
print("  theta : 0.15")
print("  sigma : 0.15 → 0.05 (anneals over training)")
print("  clip  : [0.0, 1.0]")

Ornstein-Uhlenbeck Noise:
  mu    : 0.5  (centred for [0,1] action space)
  theta : 0.15
  sigma : 0.15 → 0.05 (anneals over training)
  clip  : [0.0, 1.0]


In [ ]:
# CELL 12 — RunningNorm (for reference)
print("RunningNorm (Welford online algorithm):")
print("  Normalises observations online")
print("  normalize_batch() — vectorised, no Python loops")
print("  Clip: ±5.0")

RunningNorm (Welford online algorithm):
  Normalises observations online
  normalize_batch() — vectorised, no Python loops
  Clip: ±5.0


In [ ]:
# CELL 13 — Soft update (for reference)
print("Soft update: target = tau*source + (1-tau)*target")
print("  tau = 0.01  (1% update per step)")
print("  Applied to: target_actors and target_critics")

Soft update: target = tau*source + (1-tau)*target
  tau = 0.01  (1% update per step)
  Applied to: target_actors and target_critics


In [ ]:
# CELL 14 — Select Actions (for reference)
print("select_actions():")
print("  1. Normalize obs with RunningNorm")
print("  2. Forward pass through actor")
print("  3. Add OU noise (training) or no noise (eval)")
print("  4. Clip to [0.0, 1.0] — no action space warnings")

select_actions():
  1. Normalize obs with RunningNorm
  2. Forward pass through actor
  3. Add OU noise (training) or no noise (eval)
  4. Clip to [0.0, 1.0] — no action space warnings


In [ ]:
# CELL 15 — MADDPG Update (for reference)
print("maddpg_update() — vectorised, no Python loops:")
print("  1. Sample 512 transitions from replay buffer")
print("  2. For each agent:")
print("     a. Compute TD target using target networks")
print("     b. Update critic: MSE(Q_pred, td_target)")
print("     c. Update actor: maximize Q(s, new_actions)")
print("     d. Soft update target networks")

maddpg_update() — vectorised, no Python loops:
  1. Sample 512 transitions from replay buffer
  2. For each agent:
     a. Compute TD target using target networks
     b. Update critic: MSE(Q_pred, td_target)
     c. Update actor: maximize Q(s, new_actions)
     d. Soft update target networks


In [ ]:
# CELL 16 — Smoke test (for reference)
print("Smoke test runs inside Modal on A100:")
print("  10 environment steps with noise")
print("  Checks for action space warnings")
print("  Should print: PASSED — zero action space warnings")

Smoke test runs inside Modal on A100:
  10 environment steps with noise
  Checks for action space warnings
  Should print: PASSED — zero action space warnings


In [ ]:
# CELL 17 — Verify Modal script syntax
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'py_compile', '/content/maddpg_modal.py'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print("✅ Script syntax OK — ready to submit to Modal A100")
    print("   Script: /content/maddpg_modal.py")
    import os
    print(f"   Size  : {os.path.getsize('/content/maddpg_modal.py'):,} bytes")
else:
    print(f"❌ Syntax error: {r.stderr}")

✅ Script syntax OK — ready to submit to Modal A100
   Script: /content/maddpg_modal.py
   Size  : 17,249 bytes


In [ ]:
# CELL 18 — SUBMIT TO MODAL A100 — Main Training
# ✅ FIXED: --detach + .spawn() — Modal runs completely independently
# Submit once then close Colab — training runs until done on Modal servers
import subprocess, sys, os

print("MADDPG 1.5M — submitting to Modal A100...")
print("Using --detach + .spawn() — safe to close Colab after this cell")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "modal", "run", "--detach",
     "/content/maddpg_modal.py"],
    capture_output=True,
    text=True,
    env={**os.environ}
)

print(result.stdout)
if result.stderr:
    # Filter out the .remote() warning — we already fixed it with .spawn()
    lines = [l for l in result.stderr.split("\n")
             if "remote()" not in l and "map()" not in l]
    filtered = "\n".join(lines).strip()
    if filtered:
        print("STDERR:", filtered)

print(f"Return code: {result.returncode}")

if result.returncode == 0:
    print("\n✅ Job running on Modal A100 — fully detached")
    print("   Close this tab or shut down laptop — training continues")
    print("   Run Cell 19 anytime to check progress")
else:
    print("\n❌ Submission failed — check error above")

MADDPG 1.5M — submitting to Modal A100...
Using --detach + .spawn() — safe to close Colab after this cell
Note that running a local entrypoint in detached mode only keeps the last triggered Modal function alive after the parent process has been killed or disconnected.
✓ Initialized. View run at 
https://modal.com/apps/madhukumara1993/main/ap-xWZH9G5iLny3OhpzkwtulS
✓ Created objects.
├── 🔨 Created mount /content/maddpg_modal.py
└── 🔨 Created function train_maddpg.
Submitting MADDPG 1.5M to Modal A100...

✅ Job spawned successfully — running in background
   Function call ID: fc-01KWKG5VX8TR6C80081CH443RS
   Safe to close Colab now
   Check progress: modal volume ls maddpg-outputs MADDPG_1p5M
   Training complete when: final_actor_0.pth appears in volume
GPU  : NVIDIA A100-SXM4-40GB
CUDA : True
Save : /outputs/MADDPG_1p5M
Agents=3 | ObsDim=18 | ActionDim=5
Action space: Box(0.0, 1.0, (5,), float32)
3 actor+critic pairs created
Smoke test: PASSED — zero action space warnings
Buffer: 10

M

In [ ]:
# CELL 19 — Check Modal job status (run anytime)
import subprocess, sys, os

print("Checking Modal app status...")
r = subprocess.run(
    [sys.executable, "-m", "modal", "app", "list"],
    capture_output=True, text=True, env={**os.environ}
)
print(r.stdout if r.stdout else r.stderr)

# Also check what files are in the volume
print("\nChecking Modal volume for saved files...")
r2 = subprocess.run(
    [sys.executable, "-m", "modal", "volume", "ls",
     "maddpg-outputs", "MADDPG_1p5M"],
    capture_output=True, text=True, env={**os.environ}
)
print(r2.stdout if r2.stdout else "(volume empty or not found)")

print("\nTraining is complete when you see: final_actor_0.pth")

Checking Modal app status...
                                            Apps                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━
┃ App ID                    ┃ Description  ┃ State   ┃ Tasks ┃ Created at   ┃ St
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━
│ ap-xWZH9G5iLny3OhpzkwtulS │ maddpg-1p5m… │ stopped │ 0     │ 2026-07-03   │ 20
│                           │              │         │       │ 08:05 UTC    │ 03
└───────────────────────────┴──────────────┴─────────┴───────┴──────────────┴───


Checking Modal volume for saved files...
MADDPG_1p5M/maddpg_1p5M_curves.png
MADDPG_1p5M/final_critic_2.pth
MADDPG_1p5M/final_critic_1.pth
MADDPG_1p5M/final_critic_0.pth
MADDPG_1p5M/final_actor_2.pth
MADDPG_1p5M/final_actor_1.pth
MADDPG_1p5M/final_actor_0.pth
MADDPG_1p5M/actor_2_1500000.pth
MADDPG_1p5M/actor_1_1500000.pth
MADDPG_1p5M/actor_0_1500000.pth
MADDPG_1p5M/actor_2_1400050.pth
MADDPG_1p5M/act

In [ ]:
# CELL 20 — Greedy Evaluation
# Evaluation runs inside Modal as part of the training script
# Results are printed in Cell 18 output and saved to Modal volume
# After downloading in Cell 21, you can reload and re-evaluate locally
print("Greedy evaluation (100 episodes) runs automatically")
print("inside the Modal training job after training completes.")
print("Results are saved to Modal volume: maddpg-outputs/MADDPG_1p5M/")
print()
print("To see results: check Cell 18 output for:")
print("  Greedy Evaluation — 100 episodes")
print("  Mean Reward   : ...")
print("  Std  Reward   : ...")
print("  Max  Reward   : ...")
print("  Min  Reward   : ...")

Greedy evaluation (100 episodes) runs automatically
inside the Modal training job after training completes.
Results are saved to Modal volume: maddpg-outputs/MADDPG_1p5M/

To see results: check Cell 18 output for:
  Greedy Evaluation — 100 episodes
  Mean Reward   : ...
  Std  Reward   : ...
  Max  Reward   : ...
  Min  Reward   : ...


In [ ]:
# CELL 21 — Download results from Modal volume
# Run this AFTER training completes (Cell 18 shows Training complete)
import subprocess, sys, os

os.makedirs('/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results', exist_ok=True)

print("Downloading results from Modal volume...")
r = subprocess.run(
    [sys.executable, "-m", "modal", "volume", "get",
     "maddpg-outputs", "MADDPG_1p5M", "/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/"],
    capture_output=True, text=True, env={**os.environ}
)
print(r.stdout if r.stdout else r.stderr)

print("\nDownloaded files:")
print("=" * 58)
for f in sorted(os.listdir('/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results')):
    size = os.path.getsize(f'/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/{f}') / 1024
    print(f"  {f:45s} {size:7.1f} KB")
print("=" * 58)

✓ Finished downloading files to local!


Downloaded files:
  MADDPG_1p5M                                       4.0 KB


In [ ]:
import shutil, os

DRIVE_BASE = "/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results"
save_dir   = f"{DRIVE_BASE}/MADDPG_1p5M"
os.makedirs(save_dir, exist_ok=True)

copied = []
# Based on repeated executions and kernel state, the files are in an eight-level nested directory.
# The path is DRIVE_BASE/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/
src_dir = f"{save_dir}/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M" # Corrected path to the actual files

# Ensure this deepest directory exists, just in case.
os.makedirs(src_dir, exist_ok=True)

for f in sorted(os.listdir(src_dir)):
    if f.endswith(('.pth', '.png')):
        # The files are already in src_dir (the deepest nested directory).
        # No need to copy them, just list them and their size.
        file_path = os.path.join(src_dir, f)
        size_kb = os.path.getsize(file_path) / 1024
        copied.append((f, size_kb))

print(f"Saved {len(copied)} files → {src_dir}") # Changed save_dir to src_dir for clarity
print("=" * 58)
for fn, sz in copied:
    print(f"  {fn:45s} {sz:7.1f} KB")
print("=" * 58)

Saved 0 files → /content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M/MADDPG_1p5M


In [ ]:
# CELL 23 — Stop a running Modal job (only if you want to cancel)
# Run this ONLY if you need to stop training early
# Otherwise leave it running — it will complete automatically
import subprocess, sys, os

# First check what apps are running
r = subprocess.run(
    [sys.executable, "-m", "modal", "app", "list"],
    capture_output=True, text=True, env={**os.environ}
)
print("Running apps:")
print(r.stdout)
print("To stop: modal app stop <app-id>")
print("Or run: modal app stop maddpg-1p5m-training")

Running apps:
                                            Apps                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┳
┃ App ID                    ┃ Description ┃ State        ┃ Tasks ┃ Created at  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━╇
│ ap-xWZH9G5iLny3OhpzkwtulS │ maddpg-1p5… │ ephemeral    │ 1     │ 2026-07-03  │
│                           │             │ (detached)   │       │ 08:05 UTC   │
└───────────────────────────┴─────────────┴──────────────┴───────┴─────────────┴

To stop: modal app stop <app-id>
Or run: modal app stop maddpg-1p5m-training


In [ ]:
import subprocess, sys, os

r = subprocess.run(
    [sys.executable, "-m", "modal", "app", "list"],
    capture_output=True, text=True, env={**os.environ}
)
print(r.stdout if r.stdout else r.stderr)


                                            Apps                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┳
┃ App ID                    ┃ Description ┃ State        ┃ Tasks ┃ Created at  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━╇
│ ap-xWZH9G5iLny3OhpzkwtulS │ maddpg-1p5… │ ephemeral    │ 1     │ 2026-07-03  │
│                           │             │ (detached)   │       │ 08:05 UTC   │
└───────────────────────────┴─────────────┴──────────────┴───────┴─────────────┴



In [ ]:
import subprocess, sys, os

# Check volume for latest checkpoint files
r = subprocess.run(
    [sys.executable, "-m", "modal", "volume", "ls",
     "maddpg-outputs", "MADDPG_1p5M"],
    capture_output=True, text=True, env={**os.environ}
)
print(r.stdout if r.stdout else r.stderr)


MADDPG_1p5M/best_critic_2.pth
MADDPG_1p5M/best_critic_1.pth
MADDPG_1p5M/best_critic_0.pth
MADDPG_1p5M/best_actor_2.pth
MADDPG_1p5M/best_actor_1.pth
MADDPG_1p5M/best_actor_0.pth
MADDPG_1p5M/actor_2_50.pth
MADDPG_1p5M/actor_1_50.pth
MADDPG_1p5M/actor_0_50.pth
MADDPG_1p5M/actor_2_1000050.pth
MADDPG_1p5M/actor_1_1000050.pth
MADDPG_1p5M/actor_0_1000050.pth
MADDPG_1p5M/actor_2_1000000.pth
MADDPG_1p5M/actor_1_1000000.pth
MADDPG_1p5M/actor_0_1000000.pth
MADDPG_1p5M/actor_2_900050.pth
MADDPG_1p5M/actor_1_900050.pth
MADDPG_1p5M/actor_0_900050.pth
MADDPG_1p5M/actor_2_900000.pth
MADDPG_1p5M/actor_1_900000.pth
MADDPG_1p5M/actor_0_900000.pth
MADDPG_1p5M/actor_2_800050.pth
MADDPG_1p5M/actor_1_800050.pth
MADDPG_1p5M/actor_0_800050.pth
MADDPG_1p5M/actor_2_800000.pth
MADDPG_1p5M/actor_1_800000.pth
MADDPG_1p5M/actor_0_800000.pth
MADDPG_1p5M/actor_2_700050.pth
MADDPG_1p5M/actor_1_700050.pth
MADDPG_1p5M/actor_0_700050.pth
MADDPG_1p5M/actor_2_700000.pth
MADDPG_1p5M/actor_1_700000.pth
MADDPG_1p5M/actor_0_700

In [ ]:
import subprocess, sys, os

os.makedirs('/content/MADDPG_1p5M_results', exist_ok=True)

r = subprocess.run(
    [sys.executable, "-m", "modal", "volume", "get",
     "maddpg-outputs", "MADDPG_1p5M", "/content/MADDPG_1p5M_results/"],
    capture_output=True, text=True, env={**os.environ}
)
print(r.stdout if r.stdout else r.stderr)


✓ Finished downloading files to local!



In [31]:
import zipfile, os

SAVE_DIR = '/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_New_Final_results'
ZIP_PATH = '/content/MADDPG_New_Final_results.zip'

# Create zip
print('Creating zip file...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(SAVE_DIR)):
        full_path = os.path.join(SAVE_DIR, f)
        zf.write(full_path, f)
        print(f'  Added: {f}')

size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
print(f'\nZip created: {ZIP_PATH}')
print(f'Zip size: {size_mb:.1f} MB')

Creating zip file...
  Added: actor_0_100000.pth
  Added: actor_0_1000000.pth
  Added: actor_0_1000050.pth
  Added: actor_0_100050.pth
  Added: actor_0_1100000.pth
  Added: actor_0_1100050.pth
  Added: actor_0_1200000.pth
  Added: actor_0_1200050.pth
  Added: actor_0_1300000.pth
  Added: actor_0_1300050.pth
  Added: actor_0_1400000.pth
  Added: actor_0_1400050.pth
  Added: actor_0_1500000.pth
  Added: actor_0_200000.pth
  Added: actor_0_200050.pth
  Added: actor_0_300000.pth
  Added: actor_0_300050.pth
  Added: actor_0_400000.pth
  Added: actor_0_400050.pth
  Added: actor_0_50.pth
  Added: actor_0_500000.pth
  Added: actor_0_500050.pth
  Added: actor_0_600000.pth
  Added: actor_0_600050.pth
  Added: actor_0_700000.pth
  Added: actor_0_700050.pth
  Added: actor_0_800000.pth
  Added: actor_0_800050.pth
  Added: actor_0_900000.pth
  Added: actor_0_900050.pth
  Added: actor_1_100000.pth
  Added: actor_1_1000000.pth
  Added: actor_1_1000050.pth
  Added: actor_1_100050.pth
  Added: actor_1_1

In [32]:
# Download zip directly to your desktop
from google.colab import files
files.download('/content/MADDPG_New_Final_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# First — list the ROOT of the volume to see actual folder names
import modal, os, shutil

vol = modal.Volume.from_name("maddpg-outputs")

# First check what is inside MADDPG_1p5M
print("Contents of /MADDPG_1p5M:")
for entry in vol.listdir("/MADDPG_1p5M"):
    print(f"  {entry.path}")

Contents of /MADDPG_1p5M:
  MADDPG_1p5M/maddpg_1p5M_curves.png
  MADDPG_1p5M/final_critic_2.pth
  MADDPG_1p5M/final_critic_1.pth
  MADDPG_1p5M/final_critic_0.pth
  MADDPG_1p5M/final_actor_2.pth
  MADDPG_1p5M/final_actor_1.pth
  MADDPG_1p5M/final_actor_0.pth
  MADDPG_1p5M/actor_2_1500000.pth
  MADDPG_1p5M/actor_1_1500000.pth
  MADDPG_1p5M/actor_0_1500000.pth
  MADDPG_1p5M/actor_2_1400050.pth
  MADDPG_1p5M/actor_1_1400050.pth
  MADDPG_1p5M/actor_0_1400050.pth
  MADDPG_1p5M/actor_2_1400000.pth
  MADDPG_1p5M/actor_1_1400000.pth
  MADDPG_1p5M/actor_0_1400000.pth
  MADDPG_1p5M/actor_2_1300050.pth
  MADDPG_1p5M/actor_1_1300050.pth
  MADDPG_1p5M/actor_0_1300050.pth
  MADDPG_1p5M/actor_2_1300000.pth
  MADDPG_1p5M/actor_1_1300000.pth
  MADDPG_1p5M/actor_0_1300000.pth
  MADDPG_1p5M/best_critic_2.pth
  MADDPG_1p5M/best_critic_1.pth
  MADDPG_1p5M/best_critic_0.pth
  MADDPG_1p5M/best_actor_2.pth
  MADDPG_1p5M/best_actor_1.pth
  MADDPG_1p5M/best_actor_0.pth
  MADDPG_1p5M/actor_2_1200050.pth
  MADDPG_

In [ ]:
# Replace 'outputs' with whatever folder name appeared above
SAVE_DIR = '/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5M_New results'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('/content/MADDPG_1p5M_temp', exist_ok=True)

print('Downloading from Modal volume...')
for entry in vol.listdir("/MADDPG_1p5M"):   # ← fixed path
    filename = entry.path.split("/")[-1]
    if not filename: continue
    local_path = f'/content/MADDPG_1p5M_temp/{filename}'
    with open(local_path, 'wb') as f:
        for chunk in vol.read_file(entry.path):
            f.write(chunk)
    print(f'  Downloaded: {filename}')

# Copy to Drive
print('\nSaving to Google Drive...')
copied = []
for f in sorted(os.listdir('/content/MADDPG_1p5M_temp')):
    src = f'/content/MADDPG_1p5M_temp/{f}'
    dst = os.path.join(SAVE_DIR, f)
    shutil.copy(src, dst)
    size_kb = os.path.getsize(dst) / 1024
    copied.append((f, size_kb))

print(f'\n{"="*58}')
print(f'  Saved {len(copied)} files to Google Drive')
print(f'{"="*58}')
for fn, sz in copied:
    print(f'  {fn:45s} {sz:7.1f} KB')
print('Done ✅')

  Downloaded: maddpg_1p5M_curves.png
  Downloaded: final_critic_2.pth
  Downloaded: final_critic_1.pth
  Downloaded: final_critic_0.pth
  Downloaded: final_actor_2.pth
  Downloaded: final_actor_1.pth
  Downloaded: final_actor_0.pth
  Downloaded: actor_2_1500000.pth
  Downloaded: actor_1_1500000.pth
  Downloaded: actor_0_1500000.pth
  Downloaded: actor_2_1400050.pth
  Downloaded: actor_1_1400050.pth
  Downloaded: actor_0_1400050.pth
  Downloaded: actor_2_1400000.pth
  Downloaded: actor_1_1400000.pth
  Downloaded: actor_0_1400000.pth
  Downloaded: actor_2_1300050.pth
  Downloaded: actor_1_1300050.pth
  Downloaded: actor_0_1300050.pth
  Downloaded: actor_2_1300000.pth
  Downloaded: actor_1_1300000.pth
  Downloaded: actor_0_1300000.pth
  Downloaded: best_critic_2.pth
  Downloaded: best_critic_1.pth
  Downloaded: best_critic_0.pth
  Downloaded: best_actor_2.pth
  Downloaded: best_actor_1.pth
  Downloaded: best_actor_0.pth
  Downloaded: actor_2_1200050.pth
  Downloaded: actor_1_1200050.pth
 

In [ ]:
import modal
vol = modal.Volume.from_name("maddpg-outputs")

# List everything with full details
print("All files in /MADDPG_1p5M:")
print(f"{'Filename':50s} {'Size':>10}")
print("-" * 62)

files = list(vol.listdir("/MADDPG_1p5M"))
for entry in files:
    filename = entry.path.split("/")[-1]
    if filename:
        size_kb = entry.size / 1024 if hasattr(entry, 'size') else 0
        print(f"  {filename:48s} {size_kb:8.1f} KB")

print(f"\nTotal files: {len(files)}")

All files in /MADDPG_1p5M:
Filename                                                 Size
--------------------------------------------------------------
  maddpg_1p5M_curves.png                              169.5 KB
  final_critic_2.pth                                  336.2 KB
  final_critic_1.pth                                  336.2 KB
  final_critic_0.pth                                  336.2 KB
  final_actor_2.pth                                   289.2 KB
  final_actor_1.pth                                   289.2 KB
  final_actor_0.pth                                   289.2 KB
  actor_2_1500000.pth                                 289.2 KB
  actor_1_1500000.pth                                 289.2 KB
  actor_0_1500000.pth                                 289.2 KB
  actor_2_1400050.pth                                 289.2 KB
  actor_1_1400050.pth                                 289.2 KB
  actor_0_1400050.pth                                 289.2 KB
  actor_2_1400000.pth        

In [ ]:
import modal, os, shutil

vol = modal.Volume.from_name("maddpg-outputs")

SAVE_DIR = '/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5M_New results'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('/content/MADDPG_1p5M_temp', exist_ok=True)

print('Downloading 103 files from Modal...')
downloaded = 0
for entry in vol.listdir("/MADDPG_1p5M"):
    filename = entry.path.split("/")[-1]
    if not filename: continue
    local_path = f'/content/MADDPG_1p5M_temp/{filename}'
    with open(local_path, 'wb') as f:
        for chunk in vol.read_file(entry.path):
            f.write(chunk)
    downloaded += 1
    if downloaded % 10 == 0:
        print(f'  {downloaded}/103 files downloaded...')

print(f'  All {downloaded} files downloaded to Colab')

# Copy to Drive
print('\nCopying to Google Drive...')
copied = []
for f in sorted(os.listdir('/content/MADDPG_1p5M_temp')):
    src = f'/content/MADDPG_1p5M_temp/{f}'
    dst = os.path.join(SAVE_DIR, f)
    shutil.copy(src, dst)
    size_kb = os.path.getsize(dst) / 1024
    copied.append((f, size_kb))

print(f'\n{"="*58}')
print(f'  Saved {len(copied)} files to Google Drive')
print(f'  {SAVE_DIR}')
print(f'{"="*58}')
for fn, sz in copied:
    print(f'  {fn:45s} {sz:7.1f} KB')
print('\nDone ✅')

  10/103 files downloaded...
  20/103 files downloaded...
  30/103 files downloaded...
  40/103 files downloaded...
  50/103 files downloaded...
  60/103 files downloaded...
  70/103 files downloaded...
  80/103 files downloaded...
  90/103 files downloaded...
  100/103 files downloaded...
  All 103 files downloaded to Colab

Copying to Google Drive...

  Saved 103 files to Google Drive
  /content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5M_New results
  actor_0_100000.pth                              289.2 KB
  actor_0_1000000.pth                             289.2 KB
  actor_0_1000050.pth                             289.2 KB
  actor_0_100050.pth                              289.2 KB
  actor_0_1100000.pth                             289.2 KB
  actor_0_1100050.pth                             289.2 KB
  actor_0_1200000.pth                             289.2 KB
  actor_0_1200050.pth                             289.2 KB
  actor_0_1300000.pth                             289.2 KB

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import modal, os, shutil

vol = modal.Volume.from_name("maddpg-outputs")

# Updated save path
SAVE_DIR = '/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_New_Final_results'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('/content/MADDPG_temp', exist_ok=True)

print('Downloading from Modal volume...')
downloaded = 0
for entry in vol.listdir("/MADDPG_1p5M"):
    filename = entry.path.split("/")[-1]
    if not filename: continue
    local_path = f'/content/MADDPG_temp/{filename}'
    with open(local_path, 'wb') as f:
        for chunk in vol.read_file(entry.path):
            f.write(chunk)
    downloaded += 1
    if downloaded % 10 == 0:
        print(f'  {downloaded}/103 downloaded...')

print(f'  All {downloaded} files downloaded to Colab')

# Copy to Drive
print('\nCopying to Google Drive...')
copied = []
for f in sorted(os.listdir('/content/MADDPG_temp')):
    src = f'/content/MADDPG_temp/{f}'
    dst = os.path.join(SAVE_DIR, f)
    shutil.copy(src, dst)
    size_kb = os.path.getsize(dst) / 1024
    copied.append((f, size_kb))

print(f'\n{"="*58}')
print(f'  Saved {len(copied)} files')
print(f'  {SAVE_DIR}')
print(f'{"="*58}')
for fn, sz in copied:
    print(f'  {fn:45s} {sz:7.1f} KB')

# Verify files are in Drive
print(f'\nVerifying Drive folder...')
drive_files = os.listdir(SAVE_DIR)
print(f'  Files visible in Drive: {len(drive_files)}')
print('Done ✅')

  10/103 downloaded...
  20/103 downloaded...
  30/103 downloaded...
  40/103 downloaded...
  50/103 downloaded...
  60/103 downloaded...
  70/103 downloaded...
  80/103 downloaded...
  90/103 downloaded...
  100/103 downloaded...
  All 103 files downloaded to Colab

Copying to Google Drive...

  Saved 103 files
  /content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_New_Final_results
  actor_0_100000.pth                              289.2 KB
  actor_0_1000000.pth                             289.2 KB
  actor_0_1000050.pth                             289.2 KB
  actor_0_100050.pth                              289.2 KB
  actor_0_1100000.pth                             289.2 KB
  actor_0_1100050.pth                             289.2 KB
  actor_0_1200000.pth                             289.2 KB
  actor_0_1200050.pth                             289.2 KB
  actor_0_1300000.pth                             289.2 KB
  actor_0_1300050.pth                             289.2

In [ ]:
import os

# Force sync by reading file back from Drive
SAVE_DIR = '/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_New_Final_results'

# Check files exist
files = os.listdir(SAVE_DIR)
print(f'Files confirmed in Drive: {len(files)}')

# Force Drive to flush by reading one file
import torch
actor = torch.load(os.path.join(SAVE_DIR, 'best_actor_0.pth'), map_location='cpu')
print(f'best_actor_0.pth readable: ✅')
print(f'Layer shape: {actor["net.0.weight"].shape}')

# Force flush
import subprocess
subprocess.run(['sync'], check=True)
print('Drive sync forced ✅')
print(f'\nFull path: {SAVE_DIR}')
print('Now go to drive.google.com and refresh the page')

Files confirmed in Drive: 103
best_actor_0.pth readable: ✅
Layer shape: torch.Size([256, 18])
Drive sync forced ✅

Full path: /content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_New_Final_results
Now go to drive.google.com and refresh the page


In [34]:
# Evaluation


# MADDPG 1.5M — Greedy Evaluation with Warmed Norms
# Uses files from: MADDPG_New_Final_results

!pip install -q pettingzoo==1.24.3 gymnasium==0.29.1 supersuit==3.9.3 torch numpy

import numpy as np
import torch
import torch.nn as nn
from pettingzoo.mpe import simple_spread_v3

# ── Actor architecture (must match training exactly) ──────────────
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(18, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 5),   nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

# ── RunningNorm ───────────────────────────────────────────────────
class RunningNorm:
    def __init__(self, shape, clip=5.0, eps=1e-8):
        self.mean=np.zeros(shape,np.float64); self.var=np.ones(shape,np.float64)
        self.count=eps; self.clip=clip; self.eps=eps
    def update(self, x):
        x=np.asarray(x,np.float64); self.count+=1
        d=x-self.mean; self.mean+=d/self.count; self.var+=d*(x-self.mean)
    @property
    def std(self): return np.sqrt(self.var/self.count+self.eps)
    def normalize(self, x):
        return np.clip((np.asarray(x,np.float64)-self.mean)/self.std,
                       -self.clip,self.clip).astype(np.float32)

# ── Load models ───────────────────────────────────────────────────
MODEL_DIR = '/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_New_Final_results'
AGENTS    = ['agent_0', 'agent_1', 'agent_2']

actors = []
for i in range(3):
    a = Actor()
    a.load_state_dict(torch.load(
        f'{MODEL_DIR}/best_actor_{i}.pth', map_location='cpu'))
    a.eval()
    actors.append(a)
print('Models loaded ✅')
print(f'Layer shape: {list(actors[0].state_dict().values())[0].shape}')

# ── Environment ───────────────────────────────────────────────────
env = simple_spread_v3.parallel_env(
    N=3, local_ratio=0.5, max_cycles=50, continuous_actions=True)

# ── Step 1 — Warm up RunningNorm (500 episodes) ───────────────────
print('\nWarming up RunningNorm (500 episodes)...')
wnorms = [RunningNorm(18) for _ in range(3)]

for ep in range(500):
    obs, _ = env.reset()
    done = False
    while not done:
        for i, ag in enumerate(AGENTS):
            wnorms[i].update(obs[ag])
        actions = {}
        for i, ag in enumerate(AGENTS):
            x = torch.FloatTensor(
                wnorms[i].normalize(obs[ag])).unsqueeze(0)
            with torch.no_grad():
                a = actors[i](x).squeeze(0).numpy()
            actions[ag] = a.astype(np.float32)
        obs, _, terms, truncs, _ = env.step(actions)
        done = all(terms.values()) or all(truncs.values())
    if (ep+1) % 100 == 0:
        print(f'  Warmup {ep+1}/500 done')

print('Warmup complete ✅')

# ── Step 2 — Greedy evaluation (1500 episodes) ────────────────────
print('\nRunning greedy evaluation (1500 episodes)...')
print('-' * 58)
eval_rewards = []

for ep in range(1500):
    obs, _ = env.reset()
    ep_reward = 0.0
    done = False

    # Copy warmed norms per episode
    norms = [RunningNorm(18) for _ in range(3)]
    for i in range(3):
        norms[i].mean  = wnorms[i].mean.copy()
        norms[i].var   = wnorms[i].var.copy()
        norms[i].count = wnorms[i].count

    while not done:
        actions = {}
        for i, ag in enumerate(AGENTS):
            norms[i].update(obs[ag])
            x = torch.FloatTensor(
                norms[i].normalize(obs[ag])).unsqueeze(0)
            with torch.no_grad():
                a = actors[i](x).squeeze(0).numpy()
            actions[ag] = a.astype(np.float32)
        obs, rewards, terms, truncs, _ = env.step(actions)
        ep_reward += float(np.mean(list(rewards.values())))
        done = all(terms.values()) or all(truncs.values())

    eval_rewards.append(ep_reward)
    if (ep+1) % 300 == 0:
        print(f'  {ep+1}/1500 | Running mean: {np.mean(eval_rewards):.2f}'
              f' | Std: {np.std(eval_rewards):.2f}')

# ── Final metrics ─────────────────────────────────────────────────
print(f'\n{"="*58}')
print(f'  MADDPG 1.5M — FINAL Evaluation (1500 episodes)')
print(f'  Warmed RunningNorm (500 warmup episodes)')
print(f'{"="*58}')
print(f'  Mean Reward   : {np.mean(eval_rewards):.2f}')
print(f'  Std  Reward   : {np.std(eval_rewards):.2f}')
print(f'  Max  Reward   : {np.max(eval_rewards):.2f}')
print(f'  Min  Reward   : {np.min(eval_rewards):.2f}')
print(f'  Best Avg100   : -29.23 (from training)')
print(f'  Total Episodes: 1500')
print(f'{"="*58}')
print('\nShare these numbers to complete Phase 1 final results!')

Models loaded ✅
Layer shape: torch.Size([256, 18])

Warming up RunningNorm (500 episodes)...
  Warmup 100/500 done
  Warmup 200/500 done
  Warmup 300/500 done
  Warmup 400/500 done
  Warmup 500/500 done
Warmup complete ✅

Running greedy evaluation (1500 episodes)...
----------------------------------------------------------
  300/1500 | Running mean: -43.62 | Std: 11.23
  600/1500 | Running mean: -43.42 | Std: 10.94
  900/1500 | Running mean: -43.22 | Std: 10.55
  1200/1500 | Running mean: -43.60 | Std: 10.76
  1500/1500 | Running mean: -43.74 | Std: 10.86

  MADDPG 1.5M — FINAL Evaluation (1500 episodes)
  Warmed RunningNorm (500 warmup episodes)
  Mean Reward   : -43.74
  Std  Reward   : 10.86
  Max  Reward   : -16.12
  Min  Reward   : -94.56
  Best Avg100   : -29.23 (from training)
  Total Episodes: 1500

Share these numbers to complete Phase 1 final results!
